# Build 100 personas and run the survey
Fit only eight numbers per final respondent from their 92 permitted answers. Keep Qwen, PCA and shared scoring weights fixed. Save predictions before opening the final test. Five new questions compare allocations of a hypothetical $1 billion energy-affordability budget.

In [1]:
import os, json, hashlib, time
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
OUT = ROOT / "outputs"
def read_json(name): return json.loads((ROOT / name).read_text())
def sha(name):
    with (ROOT / name).open("rb") as f: return hashlib.file_digest(f, "sha256").hexdigest()
def save_json(name, obj): (ROOT / name).write_text(json.dumps(obj, indent=2) + "\n")
protocol = read_json("configs/final_simulation_protocol.json")
survey = read_json("configs/energy_policy_survey.json")
split = read_json("data/splits/question_split.json")
people = read_json("data/splits/respondent_split.json")["groups"]
manifest = {p["partition"]: p for p in read_json("data/splits/partition_manifest.json")["partitions"]}
items = {q["item_id"]: q for q in read_json("data/processed/question_bank.json")["items"]}
index = pd.DataFrame(read_json("outputs/question_answer_index.json"))
config = read_json("configs/embedding_model.json")
comparison = read_json("outputs/model_comparison.json")
assert protocol["selected_model"] == comparison["selected"]
inputs = {p: sha(p) for p in ["configs/final_simulation_protocol.json", "configs/energy_policy_survey.json",
          "configs/embedding_model.json", "data/splits/question_split.json", "data/splits/respondent_split.json",
          "data/splits/partition_manifest.json", "data/processed/question_bank.json", "outputs/question_answer_index.json",
          "outputs/model_comparison.json", "outputs/selected_response_model.npz", "outputs/question_answer_pca.npz",
          "outputs/question_answer_features_prior_65.npy", "outputs/response_scale_priors.json"]}
for path in ["outputs/selected_response_model.npz", "outputs/question_answer_features_prior_65.npy", "outputs/response_scale_priors.json"]:
    assert inputs[path] == comparison["files"][path]["sha256"]
with np.load(OUT / "selected_response_model.npz", allow_pickle=False) as f:
    beta_np, W_np, background = [f[k].copy() for k in ["state__beta", "state__W", "state__respondents.weight"]]
    assert str(f["architecture"]) == "prior_linear" and float(f["regularization"]) == protocol["persona_l2"]
features = np.load(OUT / "question_answer_features_prior_65.npy", allow_pickle=False)
priors = {tuple(p["option_labels"]): np.array(p["probabilities"]) for p in read_json("outputs/response_scale_priors.json")}
anchor_q = [q for group in split["anchor_groups"].values() for q in group]
test_q = split["final_test"]["columns"]
assert len(anchor_q) == 92 and len(set(anchor_q)) == 92
assert not set(anchor_q) & set(split["forbidden_input_columns_all_respondents_all_waves"])
assert not set(people["final_personas"]) & (set(people["shared_training"]) | set(people["development"]))
spec = manifest["final_anchors"]
assert sha(spec["path"]) == spec["sha256"]
inputs[spec["path"]] = spec["sha256"]
anchors = pd.read_parquet(ROOT / spec["path"])
assert anchors.pid.tolist() == people["final_personas"] and anchors.columns.tolist() == ["pid"] + anchor_q
Y = anchors[anchor_q].to_numpy(dtype=float, na_value=np.nan)
Y = np.where(np.isnan(Y), -1, Y - 1).astype(np.int64)
assert Y.shape == (100, 92) and (Y < 0).sum() == 2
for j, q in enumerate(anchor_q): assert np.all((Y[:, j] >= -1) & (Y[:, j] < len(items[q]["options"])))
pids = anchors.pid.to_numpy(dtype=str)
ids = np.array([f"P{i+1:03}" for i in range(len(pids))])
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(protocol["seed"])
print(f"{len(pids)} personas; {(Y >= 0).sum():,} anchor answers; device: {device}. No final labels loaded.")

100 personas; 9,198 anchor answers; device: mps. No final labels loaded.


## Fit the eight-number vectors
Each person starts at the background average. Only their vector changes to explain their known answers; the shared model stays frozen. The numbers are learned coordinates, not named personality traits.

In [2]:
max_options = max(len(q["options"]) for q in items.values())
E = np.zeros((len(anchor_q), max_options, 65), np.float32)
mask = np.zeros((len(anchor_q), max_options), bool)
for j, q in enumerate(anchor_q):
    rows = index[index.item_id == q].sort_values("answer_index")
    assert rows.answer_index.tolist() == list(range(len(items[q]["options"])))
    E[j, :len(rows)] = features[rows.embedding_row.to_numpy()]
    mask[j, :len(rows)] = True
beta, W = torch.tensor(beta_np, device=device), torch.tensor(W_np, device=device)
e = torch.tensor(E, device=device); valid = torch.tensor(mask, device=device)
y = torch.tensor(Y, device=device)
with torch.no_grad():
    base = e[..., -1] + e[..., :64] @ beta
    preferences = e[..., :64] @ W.T
z = nn.Parameter(torch.tensor(background.mean(0), device=device).repeat(100, 1))
optimizer = torch.optim.Adam([z], lr=protocol["persona_learning_rate"])
loss_fn = nn.CrossEntropyLoss(ignore_index=-1)
def anchor_loss():
    logits = (base + torch.einsum("bd,qkd->bqk", z, preferences)).masked_fill(~valid[None], -1e9)
    return loss_fn(logits.reshape(-1, max_options), y.reshape(-1))
history=[]
start=time.monotonic()
for step in range(protocol["persona_steps"] + 1):
    nll=anchor_loss()
    if step % 20 == 0: history.append({"step": step, "nll": float(nll.item())})
    if step == protocol["persona_steps"]: break
    loss=nll + protocol["persona_l2"] * z.square().mean()
    optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
Z=z.detach().cpu().numpy()
assert Z.shape == (100, 8) and np.isfinite(Z).all() and np.unique(Z, axis=0).shape[0] == 100
assert history[-1]["nll"] < history[0]["nll"]
assert np.array_equal(beta.cpu().numpy(), beta_np) and np.array_equal(W.cpu().numpy(), W_np)
assert not beta.requires_grad and not W.requires_grad
fit_seconds=time.monotonic()-start
np.savez(OUT / "final_persona_vectors.npz", persona_ids=ids, person_ids=pids, vectors=Z, anchor_item_ids=np.array(anchor_q),
         observed_anchor_count=(Y >= 0).sum(1), model_sha256=np.array(inputs["outputs/selected_response_model.npz"]))
# One inspectable file holds all 100 personas, with exact evidence and no invented biographies.
records=[]
for i, pid in enumerate(pids):
    evidence=[]
    for j, q in enumerate(anchor_q):
        a=int(Y[i,j]); option=items[q]["options"][a] if a >= 0 else None
        evidence.append({"item_id": q, "response_code": a+1 if a >= 0 else None,
                         "answer_text": option["label"] if option else None})
    records.append({"persona_id": str(ids[i]), "source_pid": str(pid), "vector": Z[i].tolist(),
                    "observed_anchor_count": int((Y[i] >= 0).sum()), "evidence": evidence})
save_json("outputs/final_personas.json", {"model": protocol["selected_model"], "vector_dimensions": 8,
          "question_bank": "data/processed/question_bank.json", "people": records,
          "note": "Vectors have no individually interpretable trait labels. Evidence contains only permitted anchors."})
print(f"Fitted 100 vectors in {fit_seconds:.1f}s. Anchor log loss: {history[0]['nll']:.3f} → {history[-1]['nll']:.3f}.")

Fitted 100 vectors in 0.3s. Anchor log loss: 1.391 → 1.242.


## Encode five new questions
Only the allocation changes: 100/0, 75/25, 50/50, 25/75 or 0/100 for bill credits/home improvements. All use the same five support options. These are independent ratings, not a forced choice between packages.

In [3]:
display(pd.DataFrame([{ "question": q["item_id"], "bill credits": f"{q['bill_credit_pct']}%", "home improvements": f"{q['home_improvement_pct']}%"} for q in survey["questions"]]))
print(survey["scenario"])
assert len(survey["questions"]) == 5 and not {q["item_id"] for q in survey["questions"]} & set(items)
assert all(q["question_text"] not in {i["question_text"] for i in items.values()} for q in survey["questions"])
rows=[]
for q in survey["questions"]:
    assert len(q["options"]) == 5 and q["bill_credit_pct"] + q["home_improvement_pct"] == 100
    for o in q["options"]:
        text=config["text_template"].format(instruction=config["instruction"], question_text=q["question_text"], code=o["code"], label=o["label"])
        rows.append({"embedding_row": len(rows), "item_id": q["item_id"], "answer_index": o["index"],
                     "response_code": o["code"], "candidate_answer": o["label"], "input_text": text,
                     "text_sha256": hashlib.sha256(text.encode()).hexdigest()})
from sentence_transformers import SentenceTransformer
snapshot=ROOT / config["cache_directory"] / ("models--"+config["model_id"].replace("/", "--")) / "snapshots" / config["revision"]
assert snapshot.is_dir()
model=SentenceTransformer(str(snapshot), device=str(device), local_files_only=True, trust_remote_code=False,
      model_kwargs={"dtype": torch.float32, "attn_implementation": config["attention_implementation"]})
model.eval(); model.requires_grad_(False); model.tokenizer.padding_side="left"; model.max_seq_length=config["max_sequence_length"]
assert model[1].pooling_mode == "lasttoken" and model.get_embedding_dimension() == 1024
texts=[r["input_text"] for r in rows]
tokens=model.tokenizer(texts, truncation=False, padding=False, add_special_tokens=True)
for r,t in zip(rows,tokens["input_ids"]): r["token_count"]=len(t)
assert max(r["token_count"] for r in rows) <= model.max_seq_length
parameter_versions=[p._version for p in model.parameters()]
start=time.monotonic()
with torch.inference_mode():
    novel_embeddings=model.encode(texts, batch_size=config["batch_size"], prompt="", show_progress_bar=False,
                                  convert_to_numpy=True, normalize_embeddings=True)
embedding_seconds=time.monotonic()-start
assert novel_embeddings.shape == (25, 1024) and np.isfinite(novel_embeddings).all()
assert np.allclose(np.linalg.norm(novel_embeddings, axis=1), 1, atol=1e-5)
assert parameter_versions == [p._version for p in model.parameters()]
assert not any(p.requires_grad for p in model.parameters())
with np.load(OUT / "question_answer_pca.npz", allow_pickle=False) as pca:
    assert bool(pca["whiten"])
    reduced=((novel_embeddings - pca["mean"]) @ pca["components"].T) / np.sqrt(pca["explained_variance"])
prior_columns=[]
for q in survey["questions"]:
    signature=tuple(o["label"] for o in q["options"])
    assert signature in priors
    prior_columns.extend(np.log(priors[signature]).tolist())
novel_features=np.column_stack([reduced, prior_columns]).astype(np.float32)
np.save(OUT / "novel_question_embeddings.npy", novel_embeddings, allow_pickle=False)
np.save(OUT / "novel_question_features.npy", novel_features, allow_pickle=False)
save_json("outputs/novel_question_index.json", rows)
assert novel_features.shape == (25, 65) and np.isfinite(novel_features).all()
# Releasing Qwen keeps the rest of the notebook lightweight.
del model
if device.type == "mps": torch.mps.empty_cache()
print(f"Encoded 25 candidate answers in {embedding_seconds:.1f}s; applied existing PCA without refitting.")

,question,bill credits,home improvements
0,ENERGY_2027_1,100%,0%
1,ENERGY_2027_2,75%,25%
2,ENERGY_2027_3,50%,50%
3,ENERGY_2027_4,25%,75%
4,ENERGY_2027_5,0%,100%


Consider a hypothetical federal program for 2027 with a fixed total budget of $1 billion, including delivery costs, to help low-income U.S. households with home energy costs. The money has already been set aside; choosing a package does not change total spending or taxes. Bill credits reduce recipients' electricity or heating bills during 2027. Home energy improvements, such as insulation and efficient heating equipment, are intended to reduce bills in later years. Both renters and homeowners can qualify; support for rental properties must benefit the tenant. The five packages are alternatives, and only one would be implemented. Evaluate each package independently.


/Users/louismcauliffe/code/projects/AS_Tech_Task/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Encoded 25 candidate answers in 3.6s; applied existing PCA without refitting.


## Run and freeze predictions
For each persona/question, save all five probabilities plus one seeded single-select response. Headline shares use mean probabilities to avoid random-sampling noise. Zero-vector predictions are a diagnostic comparator, not a newly selected model.

In [4]:
def score(vectors, options):
    logits=(vectors @ W_np + beta_np) @ options[:, :64].T + options[:, -1]
    exp=np.exp(logits-logits.max(1, keepdims=True))
    return exp/exp.sum(1, keepdims=True)
def existing_options(q):
    r=index[index.item_id == q].sort_values("answer_index")
    assert r.answer_index.tolist() == list(range(len(r)))
    return features[r.embedding_row.to_numpy()]
# Every test and novel question has five options.
assert all(len(items[q]["options"]) == 5 for q in test_q)
test_E=np.stack([existing_options(q) for q in test_q])
new_E=novel_features.reshape(5,5,65)
def predict_set(option_features, vectors):
    p=np.stack([score(vectors.astype(np.float64), e.astype(np.float64)) for e in option_features], axis=1)
    assert np.isfinite(p).all() and np.allclose(p.sum(-1),1) and (p >= 0).all()
    return p
test_p=predict_set(test_E,Z); novel_p=predict_set(new_E,Z)
test_zero=predict_set(test_E,np.zeros_like(Z)); novel_zero=predict_set(new_E,np.zeros_like(Z))
def population(questions):
    return np.broadcast_to(np.array([priors[tuple(o["label"] for o in q["options"])] for q in questions]), (100,len(questions),5)).copy()
test_pop=population([items[q] for q in test_q]); novel_pop=population(survey["questions"])
rng=np.random.default_rng(protocol["seed"])
def draw(p):
    cdf=p.cumsum(-1); cdf[...,-1]=1
    return (rng.random(p.shape[:2])[...,None] > cdf).sum(-1)+1
novel_sample=draw(novel_p); test_sample=draw(test_p)
assert np.all((novel_sample >= 1) & (novel_sample <= 5))
np.savez(OUT / "final_test_predictions.npz", persona_ids=ids, person_ids=pids, item_ids=np.array(test_q),
         probabilities=test_p, population_probabilities=test_pop, zero_vector_probabilities=test_zero,
         sampled_response_codes=test_sample)
np.savez(OUT / "novel_policy_predictions.npz", persona_ids=ids, person_ids=pids,
         item_ids=np.array([q["item_id"] for q in survey["questions"]]), probabilities=novel_p,
         population_probabilities=novel_pop, zero_vector_probabilities=novel_zero, sampled_response_codes=novel_sample)
responses=[]
for i, persona_id in enumerate(ids):
    for j,q in enumerate(survey["questions"]):
        code=int(novel_sample[i,j])
        responses.append({"persona_id": str(persona_id), "item_id": q["item_id"], "sampled_response_code": code,
            "sampled_answer": q["options"][code-1]["label"], "most_likely_answer": q["options"][int(novel_p[i,j].argmax())]["label"],
            "option_probabilities": novel_p[i,j].tolist()})
assert len(responses) == 500
save_json("outputs/novel_policy_responses.json", {"seed": protocol["seed"], "responses": responses,
           "note": "Synthetic single-select responses. Mean probabilities, not sampled counts, are the main aggregate estimates."})
# Independent Torch/NumPy scoring check on test wording only; no real answers loaded.
with torch.no_grad():
    et=torch.tensor(test_E,device=device)
    lt=et[..., -1] + et[...,:64] @ beta + torch.einsum("bd,qkd->bqk", torch.tensor(Z,device=device), et[...,:64] @ W.T)
    check=lt.softmax(-1).cpu().numpy()
np.testing.assert_allclose(test_p,check,atol=2e-6,rtol=2e-5)
with np.load(OUT / "final_persona_vectors.npz",allow_pickle=False) as saved: assert np.array_equal(saved["vectors"],Z)
for path,h in inputs.items(): assert sha(path) == h
outputs=["outputs/final_persona_vectors.npz", "outputs/final_personas.json", "outputs/novel_question_embeddings.npy",
         "outputs/novel_question_features.npy", "outputs/novel_question_index.json", "outputs/final_test_predictions.npz",
         "outputs/novel_policy_predictions.npz", "outputs/novel_policy_responses.json"]
seal={"created_utc":datetime.now(timezone.utc).isoformat(), "model":protocol["selected_model"], "input_sha256":inputs,
      "prediction_sha256":{p:sha(p) for p in outputs}, "personas":100, "anchor_questions":92, "observed_anchor_answers":int((Y >= 0).sum()),
      "anchor_fit_history":history, "fit_seconds":fit_seconds, "embedding_seconds":embedding_seconds,
      "final_test_answers_loaded":False, "final_test_predictions_frozen":True, "new_questions":5,
      "checks":["people and question splits disjoint", "92 anchors only", "100 distinct finite vectors", "frozen shared weights",
                "frozen Qwen and existing PCA", "normalized probabilities", "500 valid single-select responses",
                "independent score reconstruction", "saved-vector round trip", "inputs unchanged"]}
save_json("outputs/final_prediction_seal.json",seal)
print("Saved 100 personas, 500 new-question responses and frozen final-test predictions. No final-test answers opened.")

Saved 100 personas, 500 new-question responses and frozen final-test predictions. No final-test answers opened.
